# GS 2026.2 — Redes Neurais — CitizenTech

**Integrantes (nome completo + RM):**
- TODO

## 1. Problema e aplicação
- **Problema CitizenTech:** TODO
- **Público impactado e contexto:** TODO
- **Como a rede neural contribui para a solução:** TODO
- **Entradas do modelo:** TODO
- **Saída esperada:** TODO

## 0. Configuração

In [ ]:
# Esqueleto para dados TABULARES com MLP (Keras). Para imagens use CNN e para séries/sons use RNN,
# trocando apenas a função criar_modelo() e o carregamento dos dados.
ARQUIVO = "dados/base.csv"   # TODO
ALVO = "TODO_alvo"           # TODO
TIPO = "classificacao"       # "classificacao" ou "regressao"
SEMENTE = 42

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn import metrics

keras.utils.set_random_seed(SEMENTE)
CLASSIF = TIPO == "classificacao"

## 2. Dataset
- **Fonte:** TODO
- **Principais características:** TODO
- **Por que é adequado à tarefa:** TODO

In [ ]:
df = pd.read_csv(ARQUIVO)
print(df.shape)
df.info()
df.head()  # exemplos dos dados

## 3. Preparação dos dados
Decisões tomadas (tratamento, transformações, divisão): TODO

In [ ]:
df = df.dropna(subset=[ALVO]).drop_duplicates()
X = pd.get_dummies(df.drop(columns=[ALVO]), drop_first=True).astype("float32")
X = X.fillna(X.median())
y = df[ALVO]
if CLASSIF:
    codificador = LabelEncoder()
    y = codificador.fit_transform(y)
    N_CLASSES = len(codificador.classes_)
else:
    y = y.to_numpy("float32")

# treino 70% / validação 15% / teste 15%
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, random_state=SEMENTE,
                                                stratify=y if CLASSIF else None)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.1765, random_state=SEMENTE,
                                                  stratify=y_tmp if CLASSIF else None)
escalonador = StandardScaler().fit(X_train)           # ajustado só no treino
X_train, X_val, X_test = (escalonador.transform(a) for a in (X_train, X_val, X_test))
print("treino", X_train.shape, "| validação", X_val.shape, "| teste", X_test.shape)

## 4. Arquitetura da rede neural
Arquitetura escolhida (MLP/CNN/RNN) e justificativa: TODO

In [ ]:
def criar_modelo(camadas=(64, 32), dropout=0.0, lr=1e-3):
    entrada = keras.Input(shape=(X_train.shape[1],))
    x = entrada
    for n in camadas:
        x = layers.Dense(n, activation="relu")(x)
        if dropout:
            x = layers.Dropout(dropout)(x)
    if CLASSIF and N_CLASSES == 2:
        saida, perda, metricas = layers.Dense(1, activation="sigmoid")(x), "binary_crossentropy", ["accuracy"]
    elif CLASSIF:
        saida, perda, metricas = (layers.Dense(N_CLASSES, activation="softmax")(x),
                                  "sparse_categorical_crossentropy", ["accuracy"])
    else:
        saida, perda, metricas = layers.Dense(1)(x), "mse", ["mae"]
    modelo = keras.Model(entrada, saida)
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=perda, metrics=metricas)
    return modelo

criar_modelo().summary()

Como a rede processa os dados e produz o resultado: TODO

## 5. Treinamento

In [ ]:
EPOCAS, BATCH, LR = 50, 32, 1e-3    # parâmetros principais

def treinar(modelo, early_stopping=False, epocas=EPOCAS, batch=BATCH):
    cb = [keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)] if early_stopping else []
    return modelo.fit(X_train, y_train, validation_data=(X_val, y_val),
                      epochs=epocas, batch_size=batch, callbacks=cb, verbose=0)

def plotar(hist, titulo):
    chaves = [k for k in hist.history if not k.startswith("val_")]
    fig, eixos = plt.subplots(1, len(chaves), figsize=(6 * len(chaves), 4))
    for ax, k in zip(np.atleast_1d(eixos), chaves):
        ax.plot(hist.history[k], label="treino")
        ax.plot(hist.history["val_" + k], label="validação")
        ax.set_title(f"{titulo} - {k}"); ax.set_xlabel("época"); ax.legend()
    plt.show()

modelo_base = criar_modelo(lr=LR)
hist_base = treinar(modelo_base)
plotar(hist_base, "Modelo base")

## 6. Refinamento e otimização
Problemas observados (overfitting, underfitting, ruído) e estratégias aplicadas: TODO

In [ ]:
# TODO: ajustar as configurações comparadas conforme os problemas observados
configuracoes = {
    "base":                     dict(camadas=(64, 32), dropout=0.0, lr=1e-3, es=False),
    "dropout + early stopping": dict(camadas=(64, 32), dropout=0.3, lr=1e-3, es=True),
    "maior + lr menor":         dict(camadas=(128, 64, 32), dropout=0.3, lr=5e-4, es=True),
}
comparacao, modelos = [], {}
for nome, c in configuracoes.items():
    m = criar_modelo(c["camadas"], c["dropout"], c["lr"])
    h = treinar(m, early_stopping=c["es"])
    modelos[nome] = m
    linha = {"config": nome, "épocas": len(h.history["loss"])}
    linha.update({k: round(v[-1], 4) for k, v in h.history.items()})
    comparacao.append(linha)
    plotar(h, nome)
pd.DataFrame(comparacao).set_index("config")

Justificativa das alterações e se melhoraram o modelo: TODO

In [ ]:
MELHOR = "dropout + early stopping"   # TODO: escolher pela comparação acima
modelo_final = modelos[MELHOR]

## 7. Avaliação no conjunto de teste

In [ ]:
saida = modelo_final.predict(X_test, verbose=0)
if CLASSIF:
    pred = (saida.ravel() >= 0.5).astype(int) if N_CLASSES == 2 else saida.argmax(axis=1)
    print(metrics.classification_report(y_test, pred, target_names=[str(c) for c in codificador.classes_]))
    metrics.ConfusionMatrixDisplay.from_predictions(y_test, pred)
    plt.show()
else:
    pred = saida.ravel()
    print(f"RMSE={metrics.root_mean_squared_error(y_test, pred):.4f}  "
          f"MAE={metrics.mean_absolute_error(y_test, pred):.4f}  R²={metrics.r2_score(y_test, pred):.4f}")

Interpretação, pontos fortes, limitações, erros e adequação ao problema: TODO

## 8. Demonstração da aplicação
Exemplos que **não** foram usados no treino.

In [ ]:
idx = np.random.default_rng(SEMENTE).choice(len(X_test), size=5, replace=False)
saida = modelo_final.predict(X_test[idx], verbose=0)
if CLASSIF:
    pred = (saida.ravel() >= 0.5).astype(int) if N_CLASSES == 2 else saida.argmax(axis=1)
    demo = pd.DataFrame({"real": codificador.inverse_transform(np.asarray(y_test)[idx]),
                         "previsto": codificador.inverse_transform(pred)})
else:
    demo = pd.DataFrame({"real": np.asarray(y_test)[idx], "previsto": saida.ravel()})
demo

**Integração com a solução CitizenTech:** TODO
**O que ainda precisaria ser desenvolvido (simulado/não integrado):** TODO

## Documentação
- **Decisões técnicas:** TODO
- **Resultados:** TODO
- **Limitações:** TODO
- **Possibilidades de evolução:** TODO
- **Acessibilidade, segurança, proteção de dados, sustentabilidade e condições reais de uso:** TODO